# Edisi 001: Jakarta benar makin panas, atau kita saja yang mengeluh?

## Pertanyaan

Setiap tahun menjelang kemarau, keluhan yang sama muncul di linimasa: katanya Jakarta makin
panas. 76 tahun suhu harian Jakarta, dari 1950 sampai 2025. Sebelum kita mengeluh, tebak
dulu: kira-kira berapa derajat naiknya, dan apakah keluhan itu benar?

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats
from statsmodels.stats.proportion import proportion_confint, proportions_ztest

from tools import gaya

gaya.terapkan()
df = pd.read_csv("data/suhu-harian-jakarta.csv", parse_dates=["tanggal"])
df["tahun"] = df.tanggal.dt.year
df["dekade"] = (df.tahun // 10) * 10
lengkap = df[df.tahun <= 2025]
tahunan = lengkap.groupby("tahun").tmean_era5.mean().reset_index()
tahunan.columns = ["tahun", "tmean"]
print("hari:", len(df), "| rentang:", df.tanggal.min().date(), "s.d.", df.tanggal.max().date())
df.head(3)

hari: 28024 | rentang: 1950-01-01 s.d. 2026-09-22


,tanggal,tmax_era5,tmin_era5,tmean_era5,tmean_power,tmax_power,tahun,dekade
0,1950-01-01,27.7,24.4,25.9,NaN,NaN,1950,1950
1,1950-01-02,28.9,24.0,25.8,NaN,NaN,1950,1950
2,1950-01-03,28.5,23.2,25.7,NaN,NaN,1950,1950


## Data & cara ukur

Yang diukur: suhu udara harian (maksimum, minimum, rata-rata) di titik Jakarta Pusat dari
Open-Meteo Historical Weather API (reanalisis ERA5), 28.024 hari dari 1 Januari 1950 sampai
22 September 2026. Analisis memakai 76 tahun penuh, 1950 sampai 2025, karena 2026 masih
berjalan. Hasil uji kelayakan sumber: BMKG saya datangi lebih dulu, tapi data iklim
historisnya hanya terbuka lewat akun dan API key (jalur publiknya membalas 404 dan 403).

Yang hilang dari cara ukur ini, sejak awal: reanalisis itu campuran model dan pengamatan, bukan
termometer stasiun. Resolusi gridnya kasar, jadi panas kota mungkin tidak tertangkap penuh.

In [2]:
ringkas = pd.DataFrame({
    "suhu rata-rata harian (°C)": lengkap.tmean_era5,
    "suhu maksimum harian (°C)": lengkap.tmax_era5,
}).describe().T[["count", "mean", "std", "min", "max"]].round(2)
tahunan.describe().T[["count", "mean", "std", "min", "max"]].round(2)
print("bulan terpanas vs terdingin (rata-rata tmean 1950-2025):")
bulanan = lengkap.groupby(lengkap.tanggal.dt.month).tmean_era5.mean().round(2)
print(f"  Oktober {bulanan[10]} °C (terpanas) vs Januari {bulanan[1]} °C (terdingin)")
rekor = lengkap.loc[lengkap.tmax_era5.idxmax()]
print(f"  hari paling panas: tmax {rekor.tmax_era5:.1f} °C pada {rekor.tanggal.date()}")
ringkas

bulan terpanas vs terdingin (rata-rata tmean 1950-2025):
  Oktober 26.83 °C (terpanas) vs Januari 25.49 °C (terdingin)
  hari paling panas: tmax 35.8 °C pada 2024-09-07


,count,mean,std,min,max
suhu rata-rata harian (°C),27759.0,26.13,0.86,22.8,30.0
suhu maksimum harian (°C),27759.0,29.60,1.37,24.0,35.8


## Pembedahan 1: berapa derajat per dekadenya?

Cara paling sederhana: hitung satu angka suhu rata-rata untuk tiap tahun, lalu beri garis tren.
Regresi linier (OLS) menghasilkan kenaikan **0,16 °C per dekade** (CI95% 0,12 s.d. 0,19;
p < 0,001), atau total **1,17 °C** sepanjang 1950-2025. Garis tren gampang tergoda satu tahun
yang aneh, jadi saya ulang dengan Theil-Sen (median laju
kenaikan dari semua pasangan tahun, tahan terhadap pencilan): hasilnya nyaris menimpa, 0,15 °C per dekade
(CI95% 0,12 s.d. 0,19). Uji Kendall yang tidak mensyaratkan bentuk distribusi juga setuju
(tau = 0,56; p < 0,001).

Sebagai ukuran efek yang enak dibayangkan, saya bandingkan dua jendela 20 tahun: rata-rata
tahunan naik **0,82 °C** (CI95% 0,59 s.d. 1,04) dari 25,86 °C (1950-1969) ke 26,68 °C
(2006-2025), dengan Cohen's d **2,33**. Angka d sebesar itu jarang-jarang; sekali pun Anda tidak
hafal skalanya, artinya dua jendela itu nyaris tidak tumpang tindih.

In [3]:
lr = stats.linregress(tahunan.tahun, tahunan.tmean)
tcrit = stats.t.ppf(0.975, len(tahunan) - 2)
ts = stats.theilslopes(tahunan.tmean, tahunan.tahun, 0.95)
kt = stats.kendalltau(tahunan.tahun, tahunan.tmean)

a = tahunan.loc[tahunan.tahun <= 1969, "tmean"]
b = tahunan.loc[tahunan.tahun >= 2006, "tmean"]
selisih = b.mean() - a.mean()
se = np.sqrt(a.var(ddof=1) / len(a) + b.var(ddof=1) / len(b))
dfw = (a.var(ddof=1) / len(a) + b.var(ddof=1) / len(b)) ** 2 / (
    (a.var(ddof=1) / len(a)) ** 2 / (len(a) - 1) + (b.var(ddof=1) / len(b)) ** 2 / (len(b) - 1))
tc = stats.t.ppf(0.975, dfw)
sp = np.sqrt(((len(a) - 1) * a.var(ddof=1) + (len(b) - 1) * b.var(ddof=1)) / (len(a) + len(b) - 2))
tw = stats.ttest_ind(b, a, equal_var=False)

pd.DataFrame({"nilai": [
    f"{lr.slope * 10:.3f} °C/dk (CI95 {(lr.slope - tcrit * lr.stderr) * 10:.3f} s.d. {(lr.slope + tcrit * lr.stderr) * 10:.3f}) p={lr.pvalue:.2e}",
    f"{ts.slope * 10:.3f} °C/dk (CI95 {ts.low_slope * 10:.3f} s.d. {ts.high_slope * 10:.3f})",
    f"tau={kt.statistic:.3f} p={kt.pvalue:.2e}",
    f"{a.mean():.2f} → {b.mean():.2f} °C; selisih {selisih:.2f} (CI95 {selisih - tc * se:.2f} s.d. {selisih + tc * se:.2f})",
    f"t={tw.statistic:.2f} p={tw.pvalue:.2e}",
    f"d={selisih / sp:.2f}",
], }, index=["OLS tren tahunan", "Theil-Sen", "Kendall", "Welch 2006-2025 vs 1950-1969", "uji-t Welch", "Cohen's d"])

,nilai
OLS tren tahunan,0.156 °C/dk (CI95 0.124 s.d. 0.189) p=1.26e-14
Theil-Sen,0.154 °C/dk (CI95 0.117 s.d. 0.190)
Kendall,tau=0.559 p=8.75e-13
Welch 2006-2025 vs 1950-1969,25.86 → 26.68 °C; selisih 0.82 (CI95 0.59 s.d....
uji-t Welch,t=7.37 p=1.00e-08
Cohen's d,d=2.33


In [4]:
def grafik_1(mode):
    fig, ax, fs = gaya.dasar(
        mode,
        "Berapa derajat Jakarta naik per dekadenya?",
        "0,16 °C per dekade (CI95 0,12 s.d. 0,19). Total 1,17 °C sejak 1950.",
        "Open-Meteo (reanalisis ERA5), suhu rata-rata tahunan Jakarta 1950-2025", 76,
    )
    ax.scatter(tahunan.tahun, tahunan.tmean, s=26, color=gaya.INK2, zorder=3, label="rata-rata tahunan")
    x = np.array([tahunan.tahun.min(), tahunan.tahun.max()])
    ax.plot(x, lr.intercept + lr.slope * x, color=gaya.AKSEN, lw=2.4, label="garis tren (OLS)")
    ax.set_xlabel("Tahun", fontsize=9.5 * fs)
    ax.set_ylabel("Suhu rata-rata tahunan (°C)", fontsize=9.5 * fs)
    leg = ax.legend(loc="lower right", frameon=False, fontsize=9 * fs)
    for t in leg.get_texts():
        t.set_color(gaya.INK)
    ax.grid(True, axis="y")
    ax.grid(False, axis="x")
    gaya.simpan(fig, "01-deret-tren", mode)


for mode in gaya.MODE:
    grafik_1(mode)

## Pembedahan 2: hari-hari panas yang dulu langka

Rata-rata bisa menyembunyikan yang paling terasa: hari-hari kepanasan. Saya pakai
ambang **31,7 °C** untuk suhu maksimum harian, yaitu persentil ke-95 dari era dasar 1950-1979
(dulu, hanya 5% hari yang melewatinya). Lalu saya hitung berapa persen hari tiap dekade yang
melewati ambang itu, lengkap dengan interval kepercayaan Wilson-nya.

Jalannya tidak mulus: 1960-an sempat melompat ke 8,7%, 1970-an anjlok ke 3,3%. Tapi ujung
ceritanya sulit disangkal. Dekade 2020-an mencatat **21,1%** hari (CI95% 19,4 s.d. 22,8) di atas
ambang, dan kalau dibandingkan periode 2010-2025 dengan 1950-1959, hari segitu muncul **3,6 kali
lebih sering** (CI95% 3,0 s.d. 4,3; 13,3% vs 3,7%; uji proporsi z = 15,4; p < 0,001). Dulu 4
hari dari seratus, sekarang 13.

In [5]:
ambang = lengkap.loc[lengkap.tahun <= 1979, "tmax_era5"].quantile(0.95)
lengkap = lengkap.copy()
lengkap["panas"] = lengkap.tmax_era5 >= ambang
print(f"ambang P95 tmax 1950-1979: {ambang:.2f} °C")

baris = []
for d, g in lengkap.groupby("dekade"):
    k, n = int(g.panas.sum()), len(g)
    lo, hi = proportion_confint(k, n, method="wilson")
    baris.append({"dekade": f"{d}-an", "persen": 100 * k / n, "lo": 100 * lo, "hi": 100 * hi})
deka = pd.DataFrame(baris)

awal = lengkap[lengkap.dekade == 1950]
akhir = lengkap[lengkap.dekade >= 2010]
k1, n1 = int(awal.panas.sum()), len(awal)
k2, n2 = int(akhir.panas.sum()), len(akhir)
rasio = (k2 / n2) / (k1 / n1)
se_log = np.sqrt((1 - k2 / n2) / k2 + (1 - k1 / n1) / k1)
z, pz = proportions_ztest([k2, k1], [n2, n1])
deka.round(1)

ambang P95 tmax 1950-1979: 31.70 °C


,dekade,persen,lo,hi
0,1950-an,3.7,3.2,4.4
1,1960-an,8.7,7.8,9.6
2,1970-an,3.3,2.8,3.9
3,1980-an,3.8,3.3,4.5
4,1990-an,7.0,6.2,7.8
5,2000-an,9.7,8.8,10.7
6,2010-an,8.6,7.7,9.6
7,2020-an,21.1,19.4,22.8


In [6]:
def grafik_2(mode):
    fig, ax, fs = gaya.dasar(
        mode,
        "Hari panas yang dulu langka sekarang berapa sering?",
        "Dari 3,7% (1950-an) menjadi 21,1% (2020-an) hari melewati ambang 31,7 °C.",
        "Open-Meteo (reanalisis ERA5), Jakarta, 1950-2025", 27759,
    )
    warna = [gaya.INK2] * len(deka)
    warna[0] = warna[-1] = gaya.AKSEN
    ax.bar(deka.dekade, deka.persen, color=warna, width=0.66,
           yerr=[deka.persen - deka.lo, deka.hi - deka.persen], capsize=5,
           error_kw=dict(ecolor=gaya.INK2, lw=1.2))
    for i in (0, len(deka) - 1):
        ax.text(i, deka.persen.iloc[i] + 1.6, f"{deka.persen.iloc[i]:.1f}".replace(".", ",") + "%",
                ha="center", fontsize=9.5 * fs, color=gaya.AKSEN, fontweight="bold")
    ax.axhline(5, color=gaya.INK, lw=1.1, ls="--", label="1950-1979: 5% hari")
    leg = ax.legend(loc="upper left", frameon=False, fontsize=9 * fs)
    for t in leg.get_texts():
        t.set_color(gaya.INK)
    ax.set_ylabel("Hari dengan suhu maksimum ≥ 31,7 °C (% dari hari)", fontsize=9.5 * fs)
    ax.set_xlabel("Dekade", fontsize=9.5 * fs)
    ax.grid(True, axis="y")
    ax.grid(False, axis="x")
    gaya.simpan(fig, "02-hari-panas", mode)


for mode in gaya.MODE:
    grafik_2(mode)

## Uji silang: satu produk lagi, arah yang sama

Sebelum angka-angka ini keluar jurnal, ia saya bawa ke NASA POWER, produk lain (MERRA-2) yang
cara buatnya berbeda dari ERA5. Untuk jendela sama 1984-2025, NASA POWER menghasilkan laju
**0,19 °C per dekade** (p < 0,001), sementara ERA5 pada jendela itu 0,31 °C per dekade. Dua
produk sepakat soal arah dan kepastiannya, tapi berbeda soal besaran. Karena itulah klaimnya hanya di level "naik", bukan "naik persis sekian".

In [7]:
power = df.dropna(subset=["tmean_power"])
power_tahunan = power[power.tahun <= 2025].groupby("tahun").tmean_power.mean().reset_index()
lp = stats.linregress(power_tahunan.tahun, power_tahunan.tmean_power)
era5_jendela = tahunan[tahunan.tahun >= 1984]
l2 = stats.linregress(era5_jendela.tahun, era5_jendela.tmean)
print(f"NASA POWER 1984-2025: {lp.slope * 10:.3f} °C/dk (p={lp.pvalue:.2e}) | n tahun = {len(power_tahunan)}")
print(f"ERA5 jendela sama:    {l2.slope * 10:.3f} °C/dk (p={l2.pvalue:.2e})")

NASA POWER 1984-2025: 0.189 °C/dk (p=3.08e-08) | n tahun = 42
ERA5 jendela sama:    0.312 °C/dk (p=7.10e-12)


In [8]:
def kartu_teks():
    gaya.simpan(gaya.kartu("linkedin", "EDISI 001 · KOTA · OKTOBER 2026",
        "Jakarta benar makin panas,\natau kita saja\nyang mengeluh?",
        [(gaya.INK2, "Sebelum ikut-ikutan mengeluh,\nsaya cek 76 tahun termometer:\n28.024 hari suhu harian Jakarta,\ndari 1950 sampai 2025."),
         (gaya.AKSEN, "(jawabannya di dalam)")],
        "Jurnal Eksplorasi Data Keseharian"), "slide-01-pertanyaan", "linkedin")

    gaya.simpan(gaya.kartu("linkedin", "EDISI 001 · KOTA",
        "Datanya dari mana?",
        [(gaya.INK, "Open-Meteo (reanalisis ERA5),\nsuhu harian titik Jakarta Pusat,\n28.024 hari 1950-sekarang."),
         (gaya.INK2, "Reanalisis = campuran model dan\npengamatan, bukan termometer stasiun.\nNASA POWER (MERRA-2) jadi pembanding.\nBMKG ditawar lebih dulu: API\nhistorisnya butuh akun dan kunci.")],
        "Jurnal Eksplorasi Data Keseharian"), "slide-02-data", "linkedin")

    gaya.simpan(gaya.kartu("linkedin", "EDISI 001 · KOTA",
        "Temuan, batas, dan sumber",
        [(gaya.AKSEN, "Benar makin panas: suhu rata-rata\nJakarta naik 1,17 °C sejak 1950,\nhari panas 3,6 kali lebih sering."),
         (gaya.INK2, "Batas: reanalisis bukan stasiun, satu\ntitik grid, dan tren ini bercampur\ndengan panas kota serta perubahan\ntutupan lahan."),
         (gaya.INK2, "Sumber: Open-Meteo (ERA5) dan NASA\nPOWER (MERRA-2), diakses 22 Sep 2026.")],
        "Jurnal Eksplorasi Data Keseharian"), "slide-05-batas", "linkedin")


kartu_teks()
print("karousel:", sorted(p.name for p in Path("linkedin/gambar").glob("*.png")))

karousel: ['01-deret-tren.png', '02-hari-panas.png', 'slide-01-pertanyaan.png', 'slide-02-data.png', 'slide-05-batas.png']


## Temuan

> **Benar makin panas: rata-rata suhu Jakarta naik 1,17 °C sejak 1950, dan hari panas yang dulu
> langka sekarang muncul 3,6 kali lebih sering.**

Naiknya bukan desas-desus: 0,16 °C per dekade (CI95% 0,12 s.d. 0,19) dengan tiga cara hitung yang
saling menguatkan, dan sinyal hari ekstremnya lebih nyaring daripada sinyal rata-ratanya. Bonus
untuk yang suka melihat kalender: bulan terpanas Jakarta ternyata Oktober (26,83 °C), bukan Juli
yang sering disangka; dan hari paling panas di seluruh rekaman ada di 7 September 2024 (maksimum
35,8 °C), belum lama ini.

## Batas & cara reproduksi

Yang tidak boleh disimpulkan dari edisi ini: penyebab pemanasannya. Tren ini bercampur antara
sinyal iklim global, panas kota (urban heat island), dan perubahan tutupan lahan; mengaitkannya
pada satu penyebab butuh penelitian sendiri. Data adalah reanalisis berskala kasar, bukan
termometer stasiun BMKG, dan hanya satu titik grid untuk seluruh Jakarta. Edisi 012 (Agustus
2027) dijadwalkan mengulang hitungan ini dengan tambahan dua tahun data.

```bash
python3 tools/data_scraping.py
python3 -m nbconvert --to notebook --execute --inplace analisis.ipynb
```

## Sumber Data

- Open-Meteo Historical Weather API (reanalisis ERA5), titik -6,18 / 106,83, diakses 22 September 2026.
- NASA POWER (MERRA-2), titik yang sama, diakses 22 September 2026.
- Data olahan: `data/suhu-harian-jakarta.csv`. Kode pengambilan: `data_scraping.py`.
- Data mentah apa adanya: `data/mentah/`.